# Global Phase

Adapted from Qualtran v0.7.0, commit bcb17bb25360ed68ce3bc084645db3e929799389. Copyright 2023 Google LLC. SPDX-License-Identifier: Apache-2.0.

In [ ]:
from cmath import exp
from math import pi

import torch
from IPython.display import SVG, display

from examples.algorithms.binary_field_arithmetic import FieldOperation
from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum import Circuit
from flagquantum.compiler._component_trace.compound_control_resource import global_phase_cost_key
from flagquantum.compiler._component_trace.qec_gate_cost import query_qec_gate_cost
from flagquantum.compiler.registers.primitives.global_phase import global_phase_ir
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.simulation.unitary import get_unitary

## `GlobalPhase`
Applies a global phase to the circuit as a whole.

For an exponent $t$, the unitary effect is to multiply the state vector by the complex scalar
$$
(-1)^t = e^{i \pi t}
$$

The global phase of a state or circuit does not affect any observable quantity, but
keeping track of it can be a useful bookkeeping mechanism for testing circuit identities, and
the global phase becomes important when controlling an operation.

The source declares an atomic scalar with no decomposition. The example binds the scalar to an existing physical wire when appending it.

The single-qubit controlled version of `GlobalPhase` is `ZPowGate`.

#### Parameters
 - `exponent`: the exponent t of the global phase (-1)^t to apply.
 - `eps`: The precision of the rotation. This parameter is for bookkeeping and does not affect e.g. the tensor representation of this gate.


In [ ]:
signature = Signature(())

### Example Instances

In [ ]:
exponent = 0.5
dtype = torch.complex128

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([("Global phase", signature, {})])))

### Tensors and Controlled

The "tensor" of the global phase gate is just a number.

In [ ]:
coefficient = exp(1j * pi * exponent)
op = FieldOperation(1, global_phase_ir(exponent, anchor=0))
circuit = Circuit(1, dtype=dtype).append(op)
torch.testing.assert_close(
    get_unitary(circuit), coefficient * torch.eye(2, dtype=dtype), rtol=0, atol=1e-12
)
print(coefficient)

When a global phase is controlled, it is equivalent to a `ZPowGate`

In [ ]:
controlled = FieldOperation(1, global_phase_ir(exponent, controls=(0,), values=(1,)))
matrix = get_unitary(Circuit(1, dtype=dtype).append(controlled))
torch.testing.assert_close(
    matrix, torch.diag(torch.tensor([1, coefficient], dtype=dtype)), rtol=0, atol=1e-12
)
print("Controlled phase matrix")
print(matrix)

### Exact adjoint and resource counts

The scalar carrier may hold an arbitrary state. Applying the negative exponent restores the complete complex state. The precision parameter in the source is metadata for resource bookkeeping. The source charges zero QEC gates for the unbound global scalar. The native carrier uses two X instructions and two phase instructions, with zero clean work wires and zero measurements.

In [ ]:
from collections import Counter

prepared = Circuit(1, dtype=dtype).h(0).s(0)
initial = prepared.state().clone()
prepared.append(op)
torch.testing.assert_close(prepared.state(), coefficient * initial, rtol=0, atol=1e-12)
prepared.append(FieldOperation(1, global_phase_ir(-exponent, anchor=0)))
torch.testing.assert_close(prepared.state(), initial, rtol=0, atol=1e-12)
print("Scalar carrier instructions", dict(Counter(g.name for g in op.instructions)))
print("Unbound source model", query_qec_gate_cost(global_phase_cost_key(exponent)))